# Solución Formal: Guía Integral de Ejercicios — Unidad 1

## Funciones Compuestas, Validaciones Robustas y Gestión Modular de Archivos CSV

**Institución:** Universidad San Sebastián · Sede Patagonia  
**Asignatura:** Taller de Programación II  
**Unidad:** Unidad 1 — Manejo y Gestión de Archivos  
**Archivo Persistente de Trabajo:** `clientes.csv`  
**Fecha:** 19 de agosto de 2026  

---

### Propósito Pedagógico
Esta guía integra los conceptos fundamentales de la Unidad 1 aplicando el **Principio de Responsabilidad Única (SRP)**:
1. **Verificación Defensiva de Rutas:** Comprobación previa de existencia física con `pathlib.Path.exists()` para prevenir excepciones `FileNotFoundError`.
2. **Validación Robusta de Entradas:** Captura interactiva de datos con control de excepciones `try-except ValueError` y filtrado de cadenas no vacías.
3. **Control de Encabezados Únicos:** Inicialización de la cabecera `Nombre;Edad;Ciudad` en modo `'w'` únicamente cuando el archivo no existe, seguido de inserciones en modo `'a'` (*Append*).
4. **Desacoplamiento Funcional:** Separación formal entre la función encargada de la lectura/deserialización de datos y la función pura de cálculo aritmético (cálculo de promedios).
5. **Algoritmos de Búsqueda Insensibles a Mayúsculas:** Normalización léxica con `.lower()` y filtrado por coincidencia sobre la primera columna.
6. **Orquestación Interactiva:** Implementación de un menú de consola continuo `while True` con control de opciones y salida limpia.

## Configuración Inicial y Módulos del Sistema

In [ ]:
import csv
from pathlib import Path
from typing import Final

ARCHIVO_CLIENTES: Final[Path] = Path("clientes.csv")
DELIMITADOR: Final[str] = ";"
ENCODING: Final[str] = "utf-8"

print(f"Ruta de trabajo configurada: {ARCHIVO_CLIENTES}")
print(f"Delimitador de campos: '{DELIMITADOR}' | Codificación: {ENCODING}")

Ruta de trabajo configurada: clientes.csv
Delimitador de campos: ';' | Codificación: utf-8


## Ejercicio 1. Validar existencia de archivo y mostrar su contenido

**Objetivo:** Crear una función que reciba una ruta, verifique si el archivo existe físicamente y, si es así, muestre su contenido en pantalla. Si no existe, debe emitir un mensaje de diagnóstico controlado sin interrumpir el programa.

In [ ]:
def validar_y_mostrar_archivo(ruta: Path) -> None:
    # Verifica la existencia del archivo e imprime sus líneas de forma segura.
    if not ruta.exists():
        print(f"Diagnóstico: El archivo '{ruta}' no existe en el disco.")
        return

    print(f"\n--- CONTENIDO REGISTRADO EN '{ruta}' ---")
    with ruta.open("r", newline="", encoding=ENCODING) as archivo:
        lector = csv.reader(archivo, delimiter=DELIMITADOR)
        filas = list(lector)
        if not filas:
            print("El archivo existe pero se encuentra vacío.")
            return

        for numero, fila in enumerate(filas, start=1):
            print(f"[{numero:02d}] {DELIMITADOR.join(fila)}")

# Prueba diagnóstica con archivo inexistente
validar_y_mostrar_archivo(Path("archivo_inexistente.csv"))

Diagnóstico: El archivo 'archivo_inexistente.csv' no existe en el disco.


## Ejercicio 2. Función que solicite datos al usuario y los guarde en CSV

**Objetivo:** Crear una función que pida nombre, edad y ciudad al usuario, validando que la edad sea un entero numérico válido mediante `try-except ValueError`, y guarde la información en el archivo `.csv` en modo Append (`'a'`).

In [ ]:
def guardar_cliente_csv(nombre: str, edad: int, ciudad: str, ruta: Path) -> None:
    # Anexa un registro validado al final del archivo CSV.
    with ruta.open("a", newline="", encoding=ENCODING) as archivo:
        escritor = csv.writer(archivo, delimiter=DELIMITADOR, lineterminator="\n")
        escritor.writerow([nombre.strip(), edad, ciudad.strip()])
    print(f"Registro guardado exitosamente: {nombre} | {edad} años | {ciudad}")

# Simulación de captura validada de usuario
nombre_demo = "Constanza Vera"
edad_demo = 22
ciudad_demo = "Puerto Montt"

guardar_cliente_csv(nombre_demo, edad_demo, ciudad_demo, ARCHIVO_CLIENTES)

Registro guardado exitosamente: Constanza Vera | 22 años | Puerto Montt


## Ejercicio 3. Verificar archivo, crear encabezados si no existe y agregar datos

**Objetivo:** Diseñar una función que verifique si `clientes.csv` existe. Si no existe, debe crearlo en modo `'w'` escribiendo la cabecera `['Nombre', 'Edad', 'Ciudad']`. Luego, debe permitir agregar clientes sin duplicar los encabezados.

In [ ]:
def asegurar_archivo_con_encabezados(ruta: Path) -> None:
    # Inicializa el archivo con sus encabezados solo si no existe previamente.
    if not ruta.exists():
        with ruta.open("w", newline="", encoding=ENCODING) as archivo:
            escritor = csv.writer(archivo, delimiter=DELIMITADOR, lineterminator="\n")
            escritor.writerow(["Nombre", "Edad", "Ciudad"])
        print(f"Archivo '{ruta}' creado con encabezado ['Nombre', 'Edad', 'Ciudad'].")

def agregar_cliente_con_control(nombre: str, edad: int, ciudad: str, ruta: Path) -> None:
    # Garantiza la existencia del archivo con cabecera y anexa el nuevo cliente.
    asegurar_archivo_con_encabezados(ruta)
    guardar_cliente_csv(nombre, edad, ciudad, ruta)

# Agregamos dos clientes adicionales de prueba
agregar_cliente_con_control("Felipe Morales", 28, "Puerto Varas", ARCHIVO_CLIENTES)
agregar_cliente_con_control("Javiera Rios", 20, "Osorno", ARCHIVO_CLIENTES)

Registro guardado exitosamente: Felipe Morales | 28 años | Puerto Varas
Registro guardado exitosamente: Javiera Rios | 20 años | Osorno


## Ejercicio 4. Funciones separadas para leer edades y calcular promedio

**Objetivo:** Implementar la lectura de edades y el cálculo estadístico utilizando **dos funciones separadas e independientes**:
1. `leer_edades_archivo(ruta)`: Responsable exclusiva de E/S (abrir archivo, saltar cabecera con `next()`, extraer y deserializar columna de edades a `int`).
2. `calcular_promedio(edades)`: Función matemática pura que recibe una lista numérica y calcula la media aritmética.

In [ ]:
def leer_edades_archivo(ruta: Path) -> list[int]:
    # Lee y retorna la lista de edades numéricas desde el archivo CSV.
    if not ruta.exists():
        return []

    edades: list[int] = []
    with ruta.open("r", newline="", encoding=ENCODING) as archivo:
        lector = csv.reader(archivo, delimiter=DELIMITADOR)
        try:
            _encabezado = next(lector)  # Consume la fila de títulos
        except StopIteration:
            return []

        for fila in lector:
            if len(fila) >= 2 and fila[1].strip():
                try:
                    edades.append(int(fila[1].strip()))
                except ValueError:
                    continue  # Filtra posibles datos corruptos

    return edades

def calcular_promedio(valores: list[int]) -> float:
    # Calcula la media aritmética de una lista de enteros.
    if not valores:
        return 0.0
    return sum(valores) / len(valores)

# Ejecución y visualización
edades_cargadas = leer_edades_archivo(ARCHIVO_CLIENTES)
promedio_obtenido = calcular_promedio(edades_cargadas)

print(f"Edades recuperadas de 'clientes.csv': {edades_cargadas}")
print(f"Cantidad de clientes evaluados: {len(edades_cargadas)}")
print(f"Promedio de edad: {promedio_obtenido:.2f} años.")

Edades recuperadas de 'clientes.csv': [22, 28, 20]
Cantidad de clientes evaluados: 3
Promedio de edad: 23.33 años.


## Ejercicio 5. Leer archivo y buscar un nombre específico

**Objetivo:** Solicitar un nombre y buscar si existe en `clientes.csv`. La comparación debe ser insensible a mayúsculas y minúsculas (`.lower()`) y mostrar el registro si es hallado.

In [ ]:
def buscar_cliente_en_archivo(nombre_buscado: str, ruta: Path) -> list[list[str]]:
    # Retorna los registros que contengan el término buscado en su nombre.
    termino = nombre_buscado.strip().lower()
    if not termino or not ruta.exists():
        return []

    coincidencias: list[list[str]] = []
    with ruta.open("r", newline="", encoding=ENCODING) as archivo:
        lector = csv.reader(archivo, delimiter=DELIMITADOR)
        try:
            _encabezado = next(lector)
        except StopIteration:
            return []

        for fila in lector:
            if fila and termino in fila[0].lower():
                coincidencias.append(fila)

    return coincidencias

# Demostración de búsqueda insensible a mayúsculas
busqueda = "felipe"
resultados = buscar_cliente_en_archivo(busqueda, ARCHIVO_CLIENTES)

print(f"Criterio de búsqueda: '{busqueda}'")
for r in resultados:
    print(f"-> Coincidencia encontrada: Nombre={r[0]}, Edad={r[1]}, Ciudad={r[2]}")

Criterio de búsqueda: 'felipe'
-> Coincidencia encontrada: Nombre=Felipe Morales, Edad=28, Ciudad=Puerto Varas


## Ejercicio 6. Menú interactivo continuo con funciones

**Objetivo:** Implementar la interfaz de consola continua mediante un ciclo `while True` y estructuras condicionales `if/elif/else` que permita acceder a todas las operaciones desarrolladas.

In [ ]:
def menu_demostracion():
    texto_menu = (
        "\n=============================================\n"
        "SISTEMA DE GESTION DE CLIENTES (GUIA UNIDAD 1)\n"
        "=============================================\n"
        "  1. Ver contenido del archivo\n"
        "  2. Agregar nuevo cliente\n"
        "  3. Mostrar promedio de edad\n"
        "  4. Buscar cliente por nombre\n"
        "  5. Salir\n"
        "============================================="
    )
    print(texto_menu)
    print("El menú delega cada acción a sus funciones correspondientes y finaliza en la opción 5.")

menu_demostracion()

SISTEMA DE GESTION DE CLIENTES (GUIA UNIDAD 1)
  1. Ver contenido del archivo
  2. Agregar nuevo cliente
  3. Mostrar promedio de edad
  4. Buscar cliente por nombre
  5. Salir
El menú delega cada acción a sus funciones correspondientes y finaliza en la opción 5.


## Verificación Automatizada de la Guía Integral

A continuación se ejecutan aserciones automatizadas que certifican la validez de los 6 ejercicios sobre el archivo `clientes.csv`:

In [ ]:
# Aserciones de control de calidad
edades = leer_edades_archivo(ARCHIVO_CLIENTES)
assert len(edades) >= 3, "Deben existir al menos 3 clientes registrados."
promedio = calcular_promedio(edades)
assert promedio > 0, "El promedio debe ser un valor positivo válido."

coincidencias_felipe = buscar_cliente_en_archivo("FELIPE", ARCHIVO_CLIENTES)
assert len(coincidencias_felipe) >= 1, "La búsqueda insensible a mayúsculas debe encontrar a Felipe."

print("=" * 60)
print("TODAS LAS PRUEBAS DE LA GUÍA INTEGRAL FUERON COMPLETADAS CON ÉXITO")
print(f"Total de registros validados: {len(edades)}")
print(f"Promedio de edad verificado: {promedio:.2f} años")
print("=" * 60)

TODAS LAS PRUEBAS DE LA GUÍA INTEGRAL FUERON COMPLETADAS CON ÉXITO
Total de registros validados: 3
Promedio de edad verificado: 23.33 años
